In [4]:

pip install opencv-python numpy pandas scikit-learn

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 730.2 kB/s eta 0:00:17
   -- ------------------------------------- 0.8/12.6 MB 907.1 kB/s eta 0:00:14
   --- ------------------------------------ 1.0/12.6 MB 1.1 MB/s eta 0:00:11
   ---- ----------------------------------- 1.3/12.6 MB 1.0 MB/s eta 0:00:11
   ---- ----------------------------------- 1.6/12.6 MB 1.1 MB/s eta 0:00:10
   ----- ---------------------------------- 1.8/12.6 MB 1.3 MB/s eta 0:00:09
   ------ --------------------------------- 2.1/12.6 MB 1.3 MB/s eta 0:00:09
   -------- ------------------------------- 2.6/12.6 MB 1.3 MB/s eta 0:00:08
   --------- ------------------------------ 2.9/12.6 MB 1.3 MB/s eta 0:00:08
   --------- ------------------------------ 2.9/12.6 MB 1.3 MB/s eta 0:00:08
   --------- ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


## Part A

In [6]:
import os
import time
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Dataset path
DATASET_PATH = r"C:\Users\diash\OneDrive\Desktop\python\lab 06\448"


# 1. PREPROCESSING & FEATURE EXTRACTION (Baseline vs Improved)

def extract_baseline_features(gray_img):
    """
    Initial feature representation:
    - Basic intensity statistics & simple Canny edges
    """
    total_pixels = gray_img.size
    mean_val = float(np.mean(gray_img))
    contrast_val = float(np.std(gray_img))
    dark_ratio = float(np.sum(gray_img < 50) / total_pixels)
    bright_ratio = float(np.sum(gray_img > 200) / total_pixels)

    edges = cv2.Canny(gray_img, 100, 200)
    edge_density = float(np.sum(edges > 0) / total_pixels)

    return [mean_val, contrast_val, dark_ratio, bright_ratio, edge_density]


def extract_improved_features(gray_img):
    """
    Improved representation:
    - Histogram distribution (pixel spread)
    - Multi-scale / Adaptive thresholding
    - Texture / gradient magnitude (Sobel operators)
    """
    # Baseline features
    base_feats = extract_baseline_features(gray_img)

    # 1. Texture / Gradient Magnitude using Sobel filters
    grad_x = cv2.Sobel(gray_img, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray_img, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    mean_gradient = float(np.mean(grad_mag))
    std_gradient = float(np.std(grad_mag))

    # 2. 8-Bin Normalized Intensity Histogram (captures distribution)
    hist, _ = np.histogram(gray_img, bins=8, range=(0, 256), density=True)
    hist_feats = hist.tolist()

    # 3. Adaptive / Local Thresholding for thin cracks
    adaptive_thresh = cv2.adaptiveThreshold(
        gray_img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
    )
    adaptive_crack_ratio = float(np.sum(adaptive_thresh > 0) / gray_img.size)

    return base_feats + [mean_gradient, std_gradient, adaptive_crack_ratio] + hist_feats


# 2. LOAD DATASET & EXTRACT BOTH REPRESENTATIONS
X_baseline_list = []
X_improved_list = []
y_list = []

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

for root, _, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(valid_extensions):
            img_path = os.path.join(root, file)
            
            # Infer label (0 = non-crack, 1 = crack)
            path_lower = img_path.lower()
            label = 0 if any(k in path_lower for k in ["non", "no", "un"]) else 1

            # Raw image reading & preprocessing
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                continue
            
            img_resized = cv2.resize(img_bgr, (128, 128), interpolation=cv2.INTER_AREA)
            gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

            # Feature extraction
            X_baseline_list.append(extract_baseline_features(gray))
            X_improved_list.append(extract_improved_features(gray))
            y_list.append(label)

X_base = np.array(X_baseline_list)
X_impr = np.array(X_improved_list)
y = np.array(y_list)

print(f"Total Images Processed: {len(y)}")
print(f"Baseline Feature Count: {X_base.shape[1]}")
print(f"Improved Feature Count: {X_impr.shape[1]}")


# 3. EVALUATION FUNCTION (TRAIN-TEST SPLIT -> MODEL -> METRICS)
def evaluate_representation(X_data, y_data, rep_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestClassifier(n_estimators=100, random_state=42)

    # Train
    t0 = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = round((time.time() - t0) * 1000, 2)

    # Predict
    t1 = time.time()
    y_pred = model.predict(X_test_scaled)
    pred_time = round((time.time() - t1) * 1000, 2)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n==================== {rep_name} ====================")
    print(f"Accuracy:       {acc:.4f}")
    print(f"Precision:      {prec:.4f}")
    print(f"Recall:         {rec:.4f}")
    print(f"F1-Score:       {f1:.4f}")
    print(f"Train Time:     {train_time} ms")
    print(f"Inference Time: {pred_time} ms")
    print(f"Confusion Matrix:\n{cm}")


# 4. RUN BASELINE VS REPRESENTATION IMPROVEMENT
evaluate_representation(X_base, y, "Baseline Representation")
evaluate_representation(X_impr, y, "Representation Improvement")

Total Images Processed: 400
Baseline Feature Count: 5
Improved Feature Count: 16

==================== Baseline Representation ====================
Accuracy:       0.9375
Precision:      0.9268
Recall:         0.9500
F1-Score:       0.9383
Train Time:     102.54 ms
Inference Time: 5.42 ms
Confusion Matrix:
[[37  3]
 [ 2 38]]

==================== Representation Improvement ====================
Accuracy:       0.9500
Precision:      0.9500
Recall:         0.9500
F1-Score:       0.9500
Train Time:     112.52 ms
Inference Time: 11.11 ms
Confusion Matrix:
[[38  2]
 [ 2 38]]


## Part B

In [7]:
import os
import re
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# 1. LOAD DATASET & INSPECT SPAM/NON-SPAM DISTRIBUTION
CSV_PATH = "emails.csv"  # Update path if located elsewhere

df = pd.read_csv(CSV_PATH)
print("Dataset Shape:", df.shape)

# Class distribution inspection
counts = df['Prediction'].value_counts()
percentages = df['Prediction'].value_counts(normalize=True) * 100

dist_table = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': percentages.round(2)
}, index=[0, 1])
dist_table.index = ['Non-Spam (0)', 'Spam (1)']

print("\n--- Class Distribution ---")
print(dist_table)


# 2. RECONSTRUCT & PERFORM BASIC TEXT CLEANING

# Reconstruct email strings from the word frequency columns
vocab_columns = np.array(df.columns[1:-1])
frequency_matrix = df.iloc[:, 1:-1].values

df['Email_Text'] = [
    " ".join(np.repeat(vocab_columns, counts)) for counts in frequency_matrix
]

def clean_text(text):
    """Basic text cleaning: lowercase, remove non-alphabetic chars, trim whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Cleaned_Text'] = df['Email_Text'].apply(clean_text)

# 3. TRAIN-TEST SPLIT
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['Cleaned_Text'],
    df['Prediction'],
    test_size=0.20,
    random_state=42,
    stratify=df['Prediction']
)


# 4. VECTORIZATION (COUNT VECTORIZER ONLY)

# Using CountVectorizer with English stop words removed
vectorizer = CountVectorizer(stop_words='english', min_df=2)
X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

num_features = X_train_vec.shape[1]
print(f"\nNumber of Features Generated by CountVectorizer: {num_features}")



# 5. MODEL TRAINING, EVALUATION & BENCHMARK

models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

results = []

for name, clf in models.items():
    # Measure training time
    t0 = time.perf_counter()
    clf.fit(X_train_vec, y_train)
    train_time = (time.perf_counter() - t0) * 1000  # in ms

    # Measure prediction time
    t1 = time.perf_counter()
    y_pred = clf.predict(X_test_vec)
    pred_time = (time.perf_counter() - t1) * 1000  # in ms

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Train Time (ms)": round(train_time, 2),
        "Predict Time (ms)": round(pred_time, 2),
        "Confusion Matrix": cm.tolist()
    })

# Summary DataFrame
summary_df = pd.DataFrame(results)
print("\n--- Model Benchmark on CountVectorizer ---")
print(summary_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Train Time (ms)', 'Predict Time (ms)']].to_string(index=False))

for res in results:
    print(f"\nConfusion Matrix [{res['Model']}]:")
    print(np.array(res['Confusion Matrix']))

Dataset Shape: (5172, 3002)

--- Class Distribution ---
              Count  Percentage (%)
Non-Spam (0)   3672            71.0
Spam (1)       1500            29.0


C:\Users\diash\AppData\Local\Temp\ipykernel_14136\1609979820.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Email_Text'] = [
C:\Users\diash\AppData\Local\Temp\ipykernel_14136\1609979820.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Cleaned_Text'] = df['Email_Text'].apply(clean_text)



Number of Features Generated by CountVectorizer: 2731

--- Model Benchmark on CountVectorizer ---
                  Model  Accuracy  Precision  Recall  F1-Score  Train Time (ms)  Predict Time (ms)
Multinomial Naive Bayes    0.9382     0.8576  0.9433    0.8984             3.21               0.79
    Logistic Regression    0.9710     0.9355  0.9667    0.9508           180.51               0.50

Confusion Matrix [Multinomial Naive Bayes]:
[[688  47]
 [ 17 283]]

Confusion Matrix [Logistic Regression]:
[[715  20]
 [ 10 290]]


## Part C

In [8]:
import time
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. LOAD DATASET & PREPARE CLEANED TEXT
df = pd.read_csv("emails.csv")

vocab_cols = np.array(df.columns[1:-1])
word_counts = df.iloc[:, 1:-1].values
df['Email_Text'] = [" ".join(np.repeat(vocab_cols, counts)) for counts in word_counts]

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['Cleaned_Text'] = df['Email_Text'].apply(clean_text)

# Split dataset
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['Cleaned_Text'],
    df['Prediction'],
    test_size=0.20,
    random_state=42,
    stratify=df['Prediction']
)

# 2. FEATURE REPRESENTATIONS: ORIGINAL VS IMPROVED

# Original: Raw term counts with all vocab (min_df=2)
cv_orig = CountVectorizer(stop_words='english', min_df=2)
X_train_orig = cv_orig.fit_transform(X_train_raw)
X_test_orig = cv_orig.transform(X_test_raw)

# Improved: Vocabulary pruning (max_features=1500) + Binary term occurrence
cv_improved = CountVectorizer(stop_words='english', binary=True, max_features=1500)
X_train_impr = cv_improved.fit_transform(X_train_raw)
X_test_impr = cv_improved.transform(X_test_raw)

representations = [
    ("Original (Raw Frequency, All Vocab)", X_train_orig, X_test_orig),
    ("Improved (Binary Occurrence, Top 1500)", X_train_impr, X_test_impr)
]

# 3. BENCHMARK & COMPARISON

results = []
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB()
}

for rep_name, X_tr, X_te in representations:
    num_features = X_tr.shape[1]

    for model_name, model in models.items():
        # Measure training time
        t_start_train = time.perf_counter()
        model.fit(X_tr, y_train)
        train_time_ms = (time.perf_counter() - t_start_train) * 1000

        # Measure prediction time
        t_start_pred = time.perf_counter()
        preds = model.predict(X_te)
        pred_time_ms = (time.perf_counter() - t_start_pred) * 1000

        results.append({
            "Representation": rep_name,
            "Features": num_features,
            "Model": model_name,
            "Accuracy": round(accuracy_score(y_test, preds), 4),
            "Precision": round(precision_score(y_test, preds), 4),
            "Recall": round(recall_score(y_test, preds), 4),
            "F1-Score": round(f1_score(y_test, preds), 4),
            "Train Time (ms)": round(train_time_ms, 2),
            "Pred Time (ms)": round(pred_time_ms, 2)
        })

comparison_df = pd.DataFrame(results)
print("\n--- Comparative Evaluation (Part C) ---")
print(comparison_df.to_string(index=False))

C:\Users\diash\AppData\Local\Temp\ipykernel_14136\1027239076.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Email_Text'] = [" ".join(np.repeat(vocab_cols, counts)) for counts in word_counts]
C:\Users\diash\AppData\Local\Temp\ipykernel_14136\1027239076.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Cleaned_Text'] = df['Email_Text'].apply(clean_text)



--- Comparative Evaluation (Part C) ---
                        Representation  Features                   Model  Accuracy  Precision  Recall  F1-Score  Train Time (ms)  Pred Time (ms)
   Original (Raw Frequency, All Vocab)      2731     Logistic Regression    0.9710     0.9355  0.9667    0.9508           255.69            0.61
   Original (Raw Frequency, All Vocab)      2731 Multinomial Naive Bayes    0.9382     0.8576  0.9433    0.8984             2.87            0.62
Improved (Binary Occurrence, Top 1500)      1500     Logistic Regression    0.9749     0.9448  0.9700    0.9572            66.45            0.81
Improved (Binary Occurrence, Top 1500)      1500 Multinomial Naive Bayes    0.9333     0.8348  0.9600    0.8930             2.84            0.63
